# Part 2, Task 1: Edge AI Prototype

**Goal:** Train a lightweight image classification model and convert it to TensorFlow Lite for edge deployment.
**Dataset:** CIFAR-10 (Proxy for "Recyclable Items" - e.g., classifying vehicles vs animals).
**Tools:** TensorFlow, Keras, TensorFlow Lite.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import os

print(f"TensorFlow Version: {tf.__version__}")

## 1. Load and Preprocess Data

In [ ]:
# Load CIFAR-10 dataset
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.cifar10.load_data()

# Normalize pixel values to be between 0 and 1
train_images, test_images = train_images / 255.0, test_images / 255.0

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Training Data Shape: {train_images.shape}")
print(f"Test Data Shape: {test_images.shape}")

## 2. Train a Lightweight Model (CNN)

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10)
])

model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

# Train for fewer epochs to save time for this demo
history = model.fit(train_images, train_labels, epochs=5, 
                    validation_data=(test_images, test_labels))

## 3. Convert to TensorFlow Lite

In [ ]:
# Save the Keras model first
model.save('cifar10_model.h5')

# Convert the model
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the TFLite model
with open('cifar10_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("Model converted to TFLite successfully.")

## 4. Compare Model Sizes & Benefits

In [ ]:
keras_size = os.path.getsize('cifar10_model.h5') / 1024
tflite_size = os.path.getsize('cifar10_model.tflite') / 1024

print(f"Original Keras Model Size: {keras_size:.2f} KB")
print(f"TensorFlow Lite Model Size: {tflite_size:.2f} KB")
print(f"Size Reduction: {(1 - tflite_size/keras_size)*100:.2f}%")

print("\nAnalysis:")
print("The TFLite model is optimized for mobile/edge devices. It typically has a smaller footprint and faster inference time due to quantization and operator fusion, making it suitable for real-time applications like recycling sorting on a Raspberry Pi.")